# 🧪 Session 12 exercises
### Boosting, and reading a model

The lecture built boosting from one step: fit a small tree to what the forecast still gets wrong, and add a fraction of it. Then it chose the number of trees on a later block, met XGBoost and LightGBM, boosted a label, and read the models through their importance. These exercises do each of those steps yourself, starting with the six days by hand.

Three tables, all from earlier sessions. The **index table** from Session 6, with the volatility over the next 20 days as the target. The **credit table** from Session 9, with a default as the label. And the **daily prices** of the eleven instruments, for a target of your own in section J.

**Course site:** [the lecture](https://theill95.github.io/mlfin-2026/session_12/session_12.html) · [the case, in Colab](https://colab.research.google.com/github/theill95/mlfin-2026/blob/main/session_12/session_12_case.ipynb) · [the cheatsheet](https://theill95.github.io/mlfin-2026/cheatsheet.html) · [all sessions](https://theill95.github.io/mlfin-2026/)

## How to use this notebook

- Run the **setup cell** below first. It loads the three tables, makes the splits, and imports scikit-learn, XGBoost and LightGBM.
- Each exercise has a **task**, then a **code cell** for your work. Cells with `...` are blanks to fill in. Replace them with real code.
- Stuck? Open the **💡 Hint**, but only after a genuine attempt. Open the **✅ Solution** to *check* yourself, not to skip the thinking.
- Every cell runs cleanly even with the blanks still in place, so pressing **Run all** never floods you with errors.
- Most exercises stand alone. A few short runs build on each other (A4 to A6, C2 to C3, D1 to D3, D4 to D5, G4 to G5, H1 to H2, H3 to H4); the task says which earlier exercise it continues from. Section I shows five mistakes and asks you to fix them. Section J uses the table from J1, and **section K is five small cases that each start from scratch**. Eight of them ask you to draw something: A5, C3, D5, F4, G5, H2, H4 and J6.
- A few cells fit hundreds of trees or several models and take several seconds. The task says so where it matters.

**You are not expected to finish all of these.** Do what you can, and come back to the rest when you revise. Short on time? Read the hint, then the solution. A worked solution you genuinely understand is real learning too.

**Units.** The index table and the daily returns are in percent, as in the lecture. The credit table is in New Taiwan dollars, as it comes.

### Difficulty

| badge | what to expect |
|:--|:--|
| ★☆☆☆☆ | One step, straight from the lecture. You are checking that you can type it. |
| ★★☆☆☆ | The same idea on new data, or two steps in a row. Nothing to decide. |
| ★★★☆☆ | Combine two ideas, or adapt a pattern rather than copy it. |
| ★★★★☆ | You choose the approach. Several steps, and something has to be worked out before you type. |
| ★★★★★ | A genuine puzzle: an insight, or a constraint that rules out the obvious route. Always solvable with what you have. |

The stars rate the work against **this** session. A three-star task here assumes everything from Sessions 1 to 11, so it is a bigger piece of work than a three-star task in an earlier notebook.

Some exercises also carry a **revisits** tag. Those need something from an earlier session as well as today's material, and they are there on purpose: the skills are meant to accumulate.

---

## 🧰 Toolkit

New this session. Hover a name for what it does.

<span title="Boosted trees for a number. n_estimators is the number of trees M, learning_rate is nu, max_depth=1 makes each tree a stump.">`GradientBoostingRegressor`</span> · <span title="The same for a label: the trees are added on the log-odds scale, each fitted to y minus p.">`GradientBoostingClassifier`</span> · <span title="Yields the forecast after 1, 2, ..., M trees, one at a time. It is a generator: wrap it in list() to index it.">`.staged_predict()`</span> · <span title="Stops on its own once n_iter_no_change trees have not helped on a random validation_fraction of the training rows; n_estimators_ is the number kept.">`n_iter_no_change` `validation_fraction` `n_estimators_`</span> · <span title="scikit-learn&#39;s binned boosting: fast, and it has no feature_importances_.">`HistGradientBoostingRegressor`</span> · <span title="XGBoost&#39;s boosted trees, with scikit-learn&#39;s create, fit, predict pattern. reg_lambda is the penalty on the leaf values, gamma the price per leaf.">`XGBRegressor` `XGBClassifier`</span> · <span title="fit(..., eval_set=[(X, y)], verbose=False) scores every round on those rows; early_stopping_rounds=50 in the constructor stops 50 rounds after the best.">`eval_set` `early_stopping_rounds`</span> · <span title="The best round, counted from 0, so the model keeps best_iteration + 1 trees; best_score is its score on the eval_set.">`.best_iteration` `.best_score`</span> · <span title="XGBoost: the score of every round on every eval_set entry, as lists, under validation_0, validation_1, ...">`.evals_result()`</span> · <span title="A random share of the rows (subsample) or of the columns (colsample_bytree) for each tree.">`subsample` `colsample_bytree`</span> · <span title="LightGBM&#39;s boosted trees, grown leaf by leaf. num_leaves=2 makes a stump; verbose=-1 silences it.">`LGBMRegressor` `LGBMClassifier`</span> · <span title="LightGBM&#39;s early stopping: fit(..., eval_set=[...], callbacks=[early_stopping(50, verbose=False)]); then best_iteration_ trees are used.">`early_stopping` `.best_iteration_`</span> · <span title="Importance from the cuts on the training rows. XGBoost: gain by default, importance_type=&#39;weight&#39; counts cuts; LightGBM: counts by default, importance_type=&#39;gain&#39; sums the fall.">`.feature_importances_` `importance_type`</span> · <span title="The score lost when one column is shuffled, n_repeats times, on rows the model did not fit. importances_mean holds one number per column.">`permutation_importance`</span>

**Formulas**

- the boosted forecast after M trees: $\hat f_M(x) = \bar y + \nu \sum_{m=1}^{M} h_m(x)$
- each tree $h_m$ is fitted to the residuals $y_i - \hat f_{m-1}(x_i)$, the negative gradient of $\tfrac{1}{2}(y - f)^2$; for a label, to $y_i - \hat p_i$
- XGBoost's leaf value for squared error: $w = \sum r_i / (n + \lambda)$, the leaf's residuals summed, over its days plus $\lambda$
- XGBoost's gain of a cut: $G_L^2/(H_L+\lambda) + G_R^2/(H_R+\lambda) - (G_L+G_R)^2/(H_L+H_R+\lambda)$, with $G$ the sum of the gradients $f - y$ in a box and $H$, for squared error, its number of rows
- permutation importance: $I_j = s - \frac{1}{K}\sum_{k=1}^{K} s_{k,j}$

---

## ⚙️ Setup · run me first

In [ ]:
# The numbers in this notebook come from XGBoost 2.1.4, the version the lecture runs. Colab
# ships a newer XGBoost that grows slightly different trees, so there the first run installs
# 2.1.4 (about 20 seconds, once per Colab session).
import sys
import importlib.metadata
try:
    xgboost_version = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    xgboost_version = None
if xgboost_version != "2.1.4":
    if "google.colab" in sys.modules:
        import subprocess
        print("Installing XGBoost 2.1.4 for this Colab session...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost==2.1.4"], check=True)
    else:
        print(f"Note: you have XGBoost {xgboost_version}; the numbers here come from 2.1.4, "
              "which the setup guide's install line gives you.")

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.metrics import mean_squared_error, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor, XGBClassifier             # the setup guide's install line includes both
from lightgbm import LGBMRegressor, LGBMClassifier, early_stopping

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as a plain number."""
    return float(np.sqrt(mean_squared_error(actual, predicted)))


# ---- the index table from Session 6: one row per trading day, percent ----
table = pd.read_csv(data_path("market_features.csv"), parse_dates=["date"]).set_index("date")
columns = list(table.columns[:-1])      # the 19 feature columns
pair = ["vol_20d", "ret_20d"]
train = table.loc[:"2022-12-31"]
test = table.loc["2023-01-01":]
fit_rows = train.loc[:"2021-12-31"]      # 2015 to 2021, to fit on
stop_rows = train.loc["2022-01-01":]     # 2022, to choose the number of trees on
folds = TimeSeriesSplit(n_splits=5)

# ---- the credit table from Session 9: one row per borrower, no time order ----
credit = pd.read_csv(data_path("credit.csv"))
c_columns = ["limit", "age", "late_now", "months_late", "bill", "paid", "utilisation"]
c_train, c_test = train_test_split(credit, test_size=0.3, random_state=0,
                                   stratify=credit["default"])

# ---- the eleven instruments' daily returns, in percent, for section J ----
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
rets = prices.pivot(index="date", columns="ticker", values="close").pct_change().dropna() * 100

print("index :", table.shape, "  train", len(train), "(fit", len(fit_rows), "+ stop", len(stop_rows), ")  test", len(test))
print("credit:", credit.shape, "  default share", round(credit["default"].mean(), 4))
print("rets  :", rets.shape)

---

## A · Boosting the six days by hand

The lecture's six days, boosted one step at a time with NumPy, lists and loops. A4 to A6 run together.

### A1 · Start from the mean  ★☆☆☆☆  · revisits S3

The six days of the lecture are below. Boosting starts from one number for every day, the mean of `vol_next`. Compute that first forecast as an array of six equal values, the residuals `vol_next` minus the forecast, and the RSS, the sum of the squared residuals.

In [ ]:
vol_20d = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

forecast = ...
residuals = ...
rss = ...

print(forecast)
print(residuals)
print(rss)

<details>
<summary>💡 Hint</summary>

`np.full(6, vol_next.mean())` repeats the mean six times. Subtract, square and `.sum()`; wrap the RSS in `round(float(...), 3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_20d = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

forecast = np.full(6, vol_next.mean())
residuals = vol_next - forecast
rss = round(float((residuals ** 2).sum()), 3)

print(forecast)
print(residuals)
print(rss)
```

Every day starts at 0.95. The residuals run from -0.45 to 0.55 and sum to zero, as residuals from a mean always do. The RSS is 0.815: the error the first tree has to reduce.

</details>

---

### A2 · One stump on the residuals  ★★☆☆☆  · revisits S3

The first stump cuts the six days after a `vol_20d` of 0.8. A stump forecasts the mean of the residuals on each side of its cut. With a mask, compute the two means, the stump's forecast `h` for every day, the forecast after one step at a learning rate of 0.5, and the new RSS.

In [ ]:
vol_20d = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

residuals = ...
left = ...
h = ...
forecast = ...
new_rss = ...

print(h)
print(forecast)
print(new_rss)

<details>
<summary>💡 Hint 1</summary>

`left = vol_20d <= 0.8` is True for the three calm days. `np.where(left, a, b)` takes `a` where `left` is True and `b` elsewhere.

</details>

<details>
<summary>💡 Hint 2</summary>

`residuals[left].mean()` and `residuals[~left].mean()` are the stump's two values. The new forecast is `vol_next.mean() + 0.5 * h`.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_20d = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

residuals = vol_next - vol_next.mean()
left = vol_20d <= 0.8
h = np.where(left, residuals[left].mean(), residuals[~left].mean())
forecast = vol_next.mean() + 0.5 * h
new_rss = round(float(((vol_next - forecast) ** 2).sum()), 3)

print(h)
print(forecast)
print(new_rss)
```

The stump says -0.35 for the three calm days and 0.35 for the three busy ones. Half of that moves the forecast to 0.775 and 1.125, and the RSS falls from 0.815 to 0.264. This is the first step of the lecture's animation.

</details>

---

### A3 · The same, with lists  ★★☆☆☆  · revisits S2

Redo A1 without NumPy: the mean with `sum` and `len`, the residuals with a loop that appends to a list, and the RSS with a second loop. Then check the result against NumPy with `np.isclose`.

In [ ]:
vol_next = [0.5, 0.6, 0.7, 1.2, 1.5, 1.2]

mean = ...
residuals = []
for v in vol_next:
    ...
rss = 0
for r in residuals:
    ...

print(mean, residuals, rss)
print(...)

<details>
<summary>💡 Hint 1</summary>

`residuals.append(v - mean)` inside the first loop, and `rss = rss + r ** 2` inside the second.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.isclose(rss, ((np.array(vol_next) - np.mean(vol_next)) ** 2).sum())` compares the two.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_next = [0.5, 0.6, 0.7, 1.2, 1.5, 1.2]

mean = sum(vol_next) / len(vol_next)
residuals = []
for v in vol_next:
    residuals.append(v - mean)
rss = 0
for r in residuals:
    rss = rss + r ** 2

print(mean, residuals, rss)
print(np.isclose(rss, ((np.array(vol_next) - np.mean(vol_next)) ** 2).sum()))
```

0.95, the six residuals and 0.815 again, so `True`. The mean prints as `0.9500000000000001` and the residuals carry similar tails, because 0.95 has no exact binary form. That is why the check uses `np.isclose` and not `==`.

</details>

---

### A4 · One step as a function  ★★★☆☆  · revisits S2

Write `boost_step(x, y, forecast, nu)`: it fits a stump, `DecisionTreeRegressor(max_depth=1, random_state=0)`, to the residuals `y - forecast` and returns the new forecast and the stump. Then start from the mean of the six days and take six steps at a learning rate of 0.5, collecting the RSS before the first step and after each one in a list `rss_path`, rounded to three decimals.

In [ ]:
six = pd.DataFrame({'vol_20d': [0.4, 0.6, 0.8, 1.2, 1.6, 2.4]})
y6 = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])


def boost_step(x, y, forecast, nu):
    """..."""
    ...


forecast = np.full(6, y6.mean())
rss_path = [...]
for step in range(6):
    ...

print(rss_path)

<details>
<summary>💡 Hint 1</summary>

Inside the function: fit the stump on `x` and `y - forecast`, then `return forecast + nu * stump.predict(x), stump`.

</details>

<details>
<summary>💡 Hint 2</summary>

Start the list with the RSS of the mean. In the loop: `forecast, stump = boost_step(six, y6, forecast, 0.5)`, then append `round(float(((y6 - forecast) ** 2).sum()), 3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
six = pd.DataFrame({'vol_20d': [0.4, 0.6, 0.8, 1.2, 1.6, 2.4]})
y6 = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])


def boost_step(x, y, forecast, nu):
    """One step of boosting: a stump fitted to the residuals, added at the rate nu."""
    stump = DecisionTreeRegressor(max_depth=1, random_state=0)
    stump.fit(x, y - forecast)
    return forecast + nu * stump.predict(x), stump


forecast = np.full(6, y6.mean())
rss_path = [round(float(((y6 - forecast) ** 2).sum()), 3)]
for step in range(6):
    forecast, stump = boost_step(six, y6, forecast, 0.5)
    rss_path.append(round(float(((y6 - forecast) ** 2).sum()), 3))

print(rss_path)
```

`[0.815, 0.264, 0.126, 0.083, 0.059, 0.049, 0.038]`: the RSS of the lecture's animation, step by step. Each stump takes the cut that lowers the RSS most at that step, and the function is the whole procedure; the loop only repeats it. Keep `boost_step`, `six` and `y6`: A5 and A6 continue from here.

</details>

---

### A5 · Draw the path  ★★☆☆☆  · revisits S3

Draw `rss_path` from A4 against the step, 0 to 6, with a marker on every point, a label on each axis and a title.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(range(7), rss_path, marker='o')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.set_xlabel('step')`, `ax.set_ylabel('RSS')` and `ax.set_title(..., loc='left')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(7), rss_path, marker='o', color='#1c5cab')
ax.set_xlabel('step')
ax.set_ylabel('RSS')
ax.set_title('Boosting the six days at a learning rate of 0.5', loc='left')
plt.show()
```

The first step removes more than two thirds of the RSS and every later step less, because each stump can only fix what is left, and only half of it.

</details>

---

### A6 · A new day through fifty stumps  ★★★★☆  · revisits S2

Using `boost_step` from A4, take 50 steps from the mean at a learning rate of 0.5 and keep every stump in a list. A new day is forecast with the same sum: the mean plus 0.5 times each stump's forecast for that day. Forecast two new days, with a `vol_20d` of 1.0 and of 1.4, from your list, and check the result against `GradientBoostingRegressor` with the same settings.

In [ ]:
forecast = np.full(6, y6.mean())
stumps = []
for step in range(50):
    ...

new_days = pd.DataFrame({'vol_20d': [1.0, 1.4]})
by_hand = ...
library = ...

print(by_hand)
print(library)

<details>
<summary>💡 Hint 1</summary>

In the loop: `forecast, stump = boost_step(six, y6, forecast, 0.5)` and `stumps.append(stump)`.

</details>

<details>
<summary>💡 Hint 2</summary>

Start `total = np.zeros(2)` and add `stump.predict(new_days)` for every stump; then `by_hand = y6.mean() + 0.5 * total`. The library model is `GradientBoostingRegressor(n_estimators=50, learning_rate=0.5, max_depth=1, random_state=0)`, fitted on `six` and `y6`.

</details>

<details>
<summary>✅ Solution</summary>

```python
forecast = np.full(6, y6.mean())
stumps = []
for step in range(50):
    forecast, stump = boost_step(six, y6, forecast, 0.5)
    stumps.append(stump)

new_days = pd.DataFrame({'vol_20d': [1.0, 1.4]})
total = np.zeros(2)
for stump in stumps:
    total = total + stump.predict(new_days)
by_hand = y6.mean() + 0.5 * total

gbr = GradientBoostingRegressor(n_estimators=50, learning_rate=0.5, max_depth=1, random_state=0)
gbr.fit(six, y6)
library = gbr.predict(new_days)

print(by_hand)
print(library)
```

0.695 and 1.206, and the library agrees. After 50 steps the six days are fitted to within 0.006, and every cut sits halfway between two of them. A day at 1.0 is on a cut, and `<=` sends it to the side of 0.8, whose `vol_next` was 0.7; a day at 1.4 lands with 1.2, whose `vol_next` was 1.2. A boosted model is a list of trees and a sum, and between the training days its forecast jumps.

</details>

---

### A7 · Six days in five stumps  ★★★★★  · revisits S3

Boosting needed 50 steps to fit the six days to within 0.006. Five stumps can fit them exactly. Each stump below adds its value to every day above its cut, and the model starts from the first day's 0.5. Find the five values without fitting anything, so that 0.5 plus the five stumps reproduces `vol_next`, and check with `np.allclose`.

In [ ]:
vol_20d = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
cuts = [0.5, 0.7, 1.0, 1.4, 2.0]

values = ...
fit = 0.5
...

print(values)
print(fit)
print(...)

<details>
<summary>💡 Hint 1</summary>

Which days does the stump at 1.0 change, and by how much must it lift them so that the day at 1.2 is right once the first two stumps have done their part?

</details>

<details>
<summary>💡 Hint 2</summary>

Each value is the jump between two neighbouring days: `np.diff(vol_next)`. Then `for cut, value in zip(cuts, values):` with `fit = fit + value * (vol_20d > cut)` adds each stump to every day at once.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_20d = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
vol_next = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
cuts = [0.5, 0.7, 1.0, 1.4, 2.0]

values = np.diff(vol_next)
fit = 0.5
for cut, value in zip(cuts, values):
    fit = fit + value * (vol_20d > cut)

print(values)
print(fit)
print(np.allclose(fit, vol_next))
```

The values are 0.1, 0.1, 0.5, 0.3 and -0.3, the jumps between neighbouring days, and the check prints `True`. Any six numbers in a row can be written as a start and five steps, so five stumps memorise six days. Boosting took 50 steps and still missed by 0.006 because it is greedy and adds only half of each stump: it is built to approach the data slowly, not to reach it.

</details>

---

---

## B · What each tree is fitted to

The residuals are the negative gradient of the squared error, and for a label the gaps y minus p play the same part. B1 and B2 check both numerically, and B3 runs the learning rate on a single number.

### B1 · The slope of the squared error, numerically  ★★☆☆☆  · revisits S2

For one day the squared error of a forecast $f$ is $L(f) = \tfrac{1}{2}(y - f)^2$. Write `loss(y, f)` returning it for arrays, then estimate its slope at the mean forecast of the six days with a central difference, $\big(L(f + h) - L(f - h)\big) / 2h$ with $h = 0.000001$. Check that minus the slope equals the residuals.

In [ ]:
y = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
f = np.full(6, y.mean())
h = 0.000001


def loss(y, f):
    """..."""
    ...


slope = ...

print(slope)
print(...)

<details>
<summary>💡 Hint 1</summary>

`return 0.5 * (y - f) ** 2`.

</details>

<details>
<summary>💡 Hint 2</summary>

`slope = (loss(y, f + h) - loss(y, f - h)) / (2 * h)`, and `np.allclose(-slope, y - f)` is the check.

</details>

<details>
<summary>✅ Solution</summary>

```python
y = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
f = np.full(6, y.mean())
h = 0.000001


def loss(y, f):
    """Half the squared error of each forecast."""
    return 0.5 * (y - f) ** 2


slope = (loss(y, f + h) - loss(y, f - h)) / (2 * h)

print(slope)
print(np.allclose(-slope, y - f))
```

The slopes are 0.45, 0.35, 0.25, -0.25, -0.55 and -0.25, the residuals of A1 with the sign turned, and the check prints `True`. A tree fitted to the residuals is fitted to the direction in which the loss falls fastest, which is why the method is called gradient boosting.

</details>

---

### B2 · The slope of the log-loss  ★★★☆☆  · revisits S8

For a label $y$ of 0 or 1 and a log-odds $f$, the log-loss is $L(f) = -\big[y \log \sigma(f) + (1 - y) \log(1 - \sigma(f))\big]$, with $\sigma$ the sigmoid. For the four borrowers below, estimate the slope at their log-odds with a central difference, as in B1, and check that minus the slope equals $y - \sigma(f)$: the label minus the probability.

In [ ]:
y = np.array([1, 0, 0, 1])
f = np.array([0.5, -1.0, 0.2, 2.0])
h = 0.000001


def sigmoid(z):
    """..."""
    ...


def log_loss(y, f):
    """..."""
    ...


slope = ...
gap = ...

print(gap)
print(...)

<details>
<summary>💡 Hint 1</summary>

`sigmoid` returns `1 / (1 + np.exp(-z))`, and `log_loss` uses it twice, with `np.log`.

</details>

<details>
<summary>💡 Hint 2</summary>

`gap = y - sigmoid(f)`; `np.allclose(-slope, gap)` is the check.

</details>

<details>
<summary>✅ Solution</summary>

```python
y = np.array([1, 0, 0, 1])
f = np.array([0.5, -1.0, 0.2, 2.0])
h = 0.000001


def sigmoid(z):
    """The probability for a log-odds z."""
    return 1 / (1 + np.exp(-z))


def log_loss(y, f):
    """The log-loss of each label for a log-odds f."""
    return -(y * np.log(sigmoid(f)) + (1 - y) * np.log(1 - sigmoid(f)))


slope = (log_loss(y, f + h) - log_loss(y, f - h)) / (2 * h)
gap = y - sigmoid(f)

print(gap)
print(np.allclose(-slope, gap))
```

The gaps are 0.378, -0.269, -0.550 and 0.119, and the check prints `True`. A boosted classifier fits each tree to these gaps: the borrower who defaulted at a probability of 0.62 pulls the next tree up by 0.38, and the one who did not default at 0.55 pulls it down by 0.55.

</details>

---

### B3 · A learning rate on one number  ★★★☆☆  · revisits S2

Boosting with no columns at all moves one number. Start a forecast at 0 for the six days and repeat $f \leftarrow f + \nu \cdot \text{mean}(y - f)$ in a `while` loop until $f$ is within 0.001 of the mean, counting the steps, for $\nu = 0.5$ and for $\nu = 0.1$. Compare each count with the formula $\log(0.001 / \bar y) / \log(1 - \nu)$. The loop is bounded at 1,000 steps.

In [ ]:
y = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

for nu in [0.5, 0.1]:
    f = 0.0
    steps = 0
    while abs(y.mean() - f) >= 0.001 and steps < 1000:
        steps = steps + 1
        ...
    formula = ...
    print(nu, steps, formula)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, `f = f + nu * (y - f).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.log(0.001 / y.mean()) / np.log(1 - nu)`; the loop needs the next whole number above it.

</details>

<details>
<summary>✅ Solution</summary>

```python
y = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

for nu in [0.5, 0.1]:
    f = 0.0
    steps = 0
    while abs(y.mean() - f) >= 0.001 and steps < 1000:
        steps = steps + 1
        f = f + nu * (y - f).mean()
    formula = np.log(0.001 / y.mean()) / np.log(1 - nu)
    print(nu, steps, round(formula, 2))
```

10 steps at ν = 0.5 and 66 at ν = 0.1, against 9.9 and 65.1 from the formula. Each step closes the share ν of the gap that is left, so the gap after m steps is (1 − ν) to the power m of the first one. A smaller learning rate needs more steps for the same distance, roughly in proportion to 1/ν when ν is small.

</details>

---

---

## C · Boosting in scikit-learn

`GradientBoostingRegressor` on the index table: two columns, then all nineteen, the training error stump by stump, and the learning rate and the depth chosen on 2022. C2 and C3 run together.

### C1 · Two columns, a hundred stumps  ★☆☆☆☆

Fit `GradientBoostingRegressor` with 100 stumps at a learning rate of 0.1 on `pair` alone, and print its training and test RMSE.

In [ ]:
boost_pair = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)`, fitted on `train[pair]` and `train['vol_next']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
boost_pair = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
boost_pair.fit(train[pair], train['vol_next'])

print(rmse(train['vol_next'], boost_pair.predict(train[pair])))
print(rmse(test['vol_next'], boost_pair.predict(test[pair])))
```

0.535 on the training days and 0.244 on the test days. On all nineteen columns the lecture's 100 stumps scored 0.222: the other seventeen columns are worth about 0.02 here.

</details>

---

### C2 · The training error, stump by stump  ★★☆☆☆  · revisits S3

Fit 300 stumps at a learning rate of 0.1 on all of `columns`. `staged_predict` gives the forecast after each stump; compute the training RMSE after each one into a list `train_path`, and check with `np.diff` that it falls at every step.

In [ ]:
boost = ...
...
train_path = ...
falls_every_step = ...

print(falls_every_step)

<details>
<summary>💡 Hint 1</summary>

`[rmse(train['vol_next'], f) for f in boost.staged_predict(train[columns])]` builds the whole list in one line.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.diff(train_path)` holds the 299 changes, and `(np.diff(train_path) < 0).all()` is True when every one of them is negative.

</details>

<details>
<summary>✅ Solution</summary>

```python
boost = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=1, random_state=0)
boost.fit(train[columns], train['vol_next'])
train_path = [rmse(train['vol_next'], f) for f in boost.staged_predict(train[columns])]
falls_every_step = bool((np.diff(train_path) < 0).all())

print(round(train_path[0], 4), round(train_path[-1], 4))
print(falls_every_step)
```

0.676 after one stump and 0.410 after 300, and `True`: on the rows it is fitted on, every stump lowers the error, because each one is chosen to lower it. That is why the training error cannot say when to stop. Keep `train_path`: C3 draws it.

</details>

---

### C3 · Draw it on a log axis  ★★☆☆☆  · revisits S3

Draw `train_path` from C2 against the number of stumps, 1 to 300, with the x-axis on a log scale, a label on each axis and a title.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(range(1, 301), train_path)` and `ax.set_xscale('log')`.

</details>

<details>
<summary>💡 Hint 2</summary>

Label the axes, and `ax.set_title(..., loc='left')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, 301), train_path, color='#1c5cab')
ax.set_xscale('log')
ax.set_xlabel('stumps')
ax.set_ylabel('training RMSE')
ax.set_title('Training error of boosted stumps, learning rate 0.1', loc='left')
plt.show()
```

On a log axis the first ten stumps take as much width as the next ninety. The error is 0.583 after 10 stumps, 0.457 after 100 and 0.410 after 300, still falling with no floor in sight.

</details>

---

### C4 · The learning rate and the number of trees  ★★★☆☆  · revisits S2

Fit 1,000 stumps on `fit_rows`, 2015 to 2021, at each learning rate in `[0.3, 0.1, 0.03]`. For each, score every stage on `stop_rows`, 2022, find the number of stumps with the lowest 2022 RMSE, and store it in a dictionary `best_m` keyed by the rate. Print the dictionary and, for each rate, the number of stumps times the rate. This cell takes about 20 seconds.

In [ ]:
best_m = {}
for nu in [0.3, 0.1, 0.03]:
    ...

print(best_m)
for nu in best_m:
    print(nu, ...)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: fit the model, then `scores = [rmse(stop_rows['vol_next'], f) for f in model.staged_predict(stop_rows[columns])]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`int(np.argmin(scores)) + 1` is the number of stumps, since the first stage is one stump.

</details>

<details>
<summary>✅ Solution</summary>

```python
best_m = {}
for nu in [0.3, 0.1, 0.03]:
    model = GradientBoostingRegressor(n_estimators=1000, learning_rate=nu, max_depth=1, random_state=0)
    model.fit(fit_rows[columns], fit_rows['vol_next'])
    scores = [rmse(stop_rows['vol_next'], f) for f in model.staged_predict(stop_rows[columns])]
    best_m[nu] = int(np.argmin(scores)) + 1

print(best_m)
for nu in best_m:
    print(nu, round(best_m[nu] * nu, 1))
```

47 at 0.3, 219 at 0.1, 785 at 0.03; the products are 14, 22, 24. A smaller rate needs more trees, roughly in proportion, as B3 predicted. The lowest 2022 RMSEs are 0.461, 0.477 and 0.479: on this table the slower rates did not end lower, they only took longer.

</details>

---

### C5 · Deeper trees stop sooner  ★★☆☆☆

Repeat C4's search at a learning rate of 0.1 with trees of depth 1, 2 and 3, 1,000 trees each. Print the best number of trees and the best 2022 RMSE for each depth. This takes about 20 seconds.

In [ ]:
for depth in [1, 2, 3]:
    ...

<details>
<summary>💡 Hint</summary>

The same loop as C4, with `max_depth=depth` and `learning_rate=0.1`; print `int(np.argmin(scores)) + 1` and `round(min(scores), 4)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for depth in [1, 2, 3]:
    model = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=depth, random_state=0)
    model.fit(fit_rows[columns], fit_rows['vol_next'])
    scores = [rmse(stop_rows['vol_next'], f) for f in model.staged_predict(stop_rows[columns])]
    print(depth, int(np.argmin(scores)) + 1, round(min(scores), 4))
```

219 stumps, 69 trees of depth 2 and 41 of depth 3, with 2022 RMSEs of 0.477, 0.503 and 0.554. Deeper trees fit more per step, so they reach their best sooner, and on 2022 they never reach the stumps' level.

</details>

---

---

## D · How many trees

Early stopping is a choice made on validation rows. These rebuild XGBoost's rule by hand, and compare the 2022 block with the folds. D1 to D3, and D4 to D5, run together.

### D1 · Every round, scored on 2022  ★★☆☆☆

Fit `XGBRegressor` with 600 stumps at a learning rate of 0.1 on `fit_rows`, with `stop_rows` as `eval_set` and no early stopping. `evals_result()` returns the 2022 RMSE after every round under `['validation_0']['rmse']`. Store that list as `scores`, and print its length, the round with the lowest score, counted from 0, and that score.

In [ ]:
xgb600 = ...
...
scores = ...

print(...)

<details>
<summary>💡 Hint 1</summary>

`xgb600.fit(fit_rows[columns], fit_rows['vol_next'], eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`int(np.argmin(scores))` is the round, and `min(scores)` the score.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb600 = XGBRegressor(n_estimators=600, learning_rate=0.1, max_depth=1, random_state=0)
xgb600.fit(fit_rows[columns], fit_rows['vol_next'],
           eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)
scores = xgb600.evals_result()['validation_0']['rmse']

print(len(scores), int(np.argmin(scores)), round(min(scores), 4))
```

600 scores, and the lowest is 0.464, at round 243; the first round scores 0.681. Keep `scores`: D2 and D3 stop on it by hand.

</details>

---

### D2 · Early stopping, by hand  ★★★★☆  · revisits S2

XGBoost's `early_stopping_rounds=50` stops once 50 rounds have passed without a new lowest score, and keeps the best round. Write that rule with a `for` loop over `scores` from D1: keep the best score so far and its round, and `break` as soon as the current round is 50 past the best. Then fit the stopped model as the lecture did, and check that its `best_iteration` equals your best round.

In [ ]:
best_score = ...
best_round = ...
...
stopped_at = ...

print('best round', best_round, ' stopped at', stopped_at)

xgb_stop = ...
...
print(...)

<details>
<summary>💡 Hint 1</summary>

Start from `best_score = scores[0]` and `best_round = 0`, and loop with `for i, s in enumerate(scores):`. Inside, `if s < best_score:` updates both, and `if i - best_round >= 50: break` stops. After the loop, `stopped_at = i`.

</details>

<details>
<summary>💡 Hint 2</summary>

`XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1, early_stopping_rounds=50, random_state=0)`, fitted with the same `eval_set`; compare `xgb_stop.best_iteration == best_round`.

</details>

<details>
<summary>✅ Solution</summary>

```python
best_score = scores[0]
best_round = 0
for i, s in enumerate(scores):
    if s < best_score:
        best_score = s
        best_round = i
    if i - best_round >= 50:
        break
stopped_at = i

print('best round', best_round, ' stopped at', stopped_at)

xgb_stop = XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1,
                        early_stopping_rounds=50, random_state=0)
xgb_stop.fit(fit_rows[columns], fit_rows['vol_next'],
             eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)
print(xgb_stop.best_iteration == best_round)
```

The loop stops at round 293 with the best at 243, and XGBoost agrees: `True`. Early stopping fits 50 rounds it will not keep, and `predict` then uses the first 244 trees, rounds 0 to 243.

</details>

---

### D3 · Patience as an argument  ★★★☆☆  · revisits S2

Turn D2's loop into a function `stop_round(scores, patience=50)` that returns the best round and the round it stopped at. Call it on `scores` from D1 with a patience of 5, 20, 50 and 200, and print the best round and its 2022 score for each.

In [ ]:
def stop_round(scores, patience=50):
    """..."""
    ...


for patience in [5, 20, 50, 200]:
    ...

<details>
<summary>💡 Hint 1</summary>

Inside the function, D2's loop with `patience` in place of 50, returning `best_round, i` at the `break`. After the loop, `return best_round, len(scores) - 1` covers a run that never stops.

</details>

<details>
<summary>💡 Hint 2</summary>

In the loop below: `best, stopped = stop_round(scores, patience)` and `print(patience, best, stopped, round(scores[best], 4))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def stop_round(scores, patience=50):
    """The best round, and the round at which patience ran out."""
    best_score = scores[0]
    best_round = 0
    for i, s in enumerate(scores):
        if s < best_score:
            best_score = s
            best_round = i
        if i - best_round >= patience:
            return best_round, i
    return best_round, len(scores) - 1


for patience in [5, 20, 50, 200]:
    best, stopped = stop_round(scores, patience)
    print(patience, best, stopped, round(scores[best], 4))
```

With a patience of 5 or 20 the best round is 50, at 0.481; with 50 or 200 it is 243, at 0.464. The scores rise after round 50 and fall below it only much later, so too little patience stops in the first dip. The default argument makes 50 the value you get when you do not say.

</details>

---

### D4 · The folds choose fewer trees  ★★★☆☆  · revisits S5

Score boosted stumps at a learning rate of 0.1 on the time-series `folds` for `n_estimators` in `[10, 25, 50, 100, 200, 400]`, collect the mean fold RMSE in a dictionary `fold_rmse`, and print the number of trees the folds prefer. The 2022 block preferred 219 in C4. This takes about 20 seconds.

In [ ]:
fold_rmse = {}
for n in [10, 25, 50, 100, 200, 400]:
    ...

print(fold_rmse)
print(...)

<details>
<summary>💡 Hint 1</summary>

`-cross_val_score(GradientBoostingRegressor(n_estimators=n, learning_rate=0.1, max_depth=1, random_state=0), train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error').mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(fold_rmse, key=fold_rmse.get)` is the key with the smallest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
fold_rmse = {}
for n in [10, 25, 50, 100, 200, 400]:
    model = GradientBoostingRegressor(n_estimators=n, learning_rate=0.1, max_depth=1, random_state=0)
    fold_rmse[n] = float(-cross_val_score(model, train[columns], train['vol_next'], cv=folds,
                                          scoring='neg_root_mean_squared_error').mean())

print({n: round(v, 4) for n, v in fold_rmse.items()})
print(min(fold_rmse, key=fold_rmse.get))
```

The folds prefer 25 stumps, at 0.670, and the error rises steadily after that, to 0.776 at 400. Ridge with an alpha of 1000 scores 0.560 on the same folds. On the early folds the fitting rows are few, so more trees overfit them quickly. Two validation choices that both respect time disagree here by a factor of about nine. Keep `fold_rmse`: D5 draws it.

</details>

---

### D5 · Draw the fold curve  ★★☆☆☆  · revisits S3

Draw `fold_rmse` from D4 against the number of trees on a log x-axis, mark the lowest point, and add a dashed horizontal line at ridge's fold RMSE, 0.560, with a legend.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
...
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(list(fold_rmse.keys()), list(fold_rmse.values()), marker='o', label='boosted stumps')` and `ax.set_xscale('log')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.axhline(0.560, linestyle='--', color='grey', label='ridge')` for ridge, `ax.scatter` for the lowest point, then `ax.legend()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
best = min(fold_rmse, key=fold_rmse.get)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(fold_rmse.keys()), list(fold_rmse.values()), marker='o', color='#1c5cab', label='boosted stumps')
ax.scatter([best], [fold_rmse[best]], s=80, color='#b3402f', zorder=3)
ax.axhline(0.560, linestyle='--', color='grey', label='ridge')
ax.set_xscale('log')
ax.set_xlabel('trees')
ax.set_ylabel('mean fold RMSE')
ax.legend()
ax.set_title('Boosted stumps on the time-series folds', loc='left')
plt.show()
```

The curve sits above ridge's line everywhere. On these folds boosted stumps are never better than ridge, and they get worse with every tree after 25.

</details>

---

### D6 · Stopping on random days, five times  ★★☆☆☆  · revisits S2

`GradientBoostingRegressor` stops on its own with `n_iter_no_change=10` and `validation_fraction=0.1`, holding out a random tenth of the training days. Fit it with up to 5,000 stumps at a learning rate of 0.1 for `random_state` 0 to 4, and print the number of trees each keeps, `n_estimators_`. This takes about 15 seconds.

In [ ]:
for seed in range(5):
    ...

<details>
<summary>💡 Hint</summary>

`GradientBoostingRegressor(n_estimators=5000, learning_rate=0.1, max_depth=1, validation_fraction=0.1, n_iter_no_change=10, random_state=seed)`, fitted on `train[columns]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for seed in range(5):
    model = GradientBoostingRegressor(n_estimators=5000, learning_rate=0.1, max_depth=1,
                                      validation_fraction=0.1, n_iter_no_change=10, random_state=seed)
    model.fit(train[columns], train['vol_next'])
    print(seed, model.n_estimators_)
```

342, 248, 177, 523 and 430 trees: the number depends on which tenth was drawn, by a factor of three. The held-out days have neighbours among the fitting days, so every run stops late and none agrees with another, which is why the lecture stopped on a later block instead.

</details>

---

---

## E · XGBoost and LightGBM

The two libraries on the same table: their numbers against scikit-learn's, the penalty inside XGBoost's leaf values, their speed, and settings the lecture listed as sometimes worth changing.

### E1 · The same stumps in XGBoost  ★☆☆☆☆

Fit `XGBRegressor` with 100 stumps at a learning rate of 0.1 on `pair`, and print its test RMSE.

In [ ]:
xgb_pair = ...
...

print(...)

<details>
<summary>💡 Hint</summary>

`XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)`, fitted on `train[pair]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_pair = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
xgb_pair.fit(train[pair], train['vol_next'])

print(rmse(test['vol_next'], xgb_pair.predict(test[pair])))
```

0.242, against 0.244 for scikit-learn's 100 stumps on the same two columns (C1). XGBoost searches 256 bins per column and puts a small penalty on each leaf, so the two are close and not identical.

</details>

---

### E2 · The leaf values from the formula  ★★★★☆  · revisits S3

For squared error, XGBoost gives a leaf the value $\sum r_i / (n + \lambda)$: the sum of its residuals over its number of days plus $\lambda$, which is `reg_lambda`. Fit one stump on the six days with `learning_rate=1`, `base_score` set to the mean of `y6` and `reg_lambda` of 0, 1 and 10 in turn; its cut falls after 0.8 each time. Compute the two leaf values from the formula and check the model's forecasts against them.

In [ ]:
six = pd.DataFrame({'vol_20d': [0.4, 0.6, 0.8, 1.2, 1.6, 2.4]})
y6 = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

left = ...
residuals = ...

for lam in [0, 1, 10]:
    one_tree = ...
    ...
    w_left = ...
    w_right = ...
    print(lam, w_left, w_right, ...)

<details>
<summary>💡 Hint 1</summary>

`XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1, base_score=y6.mean(), reg_lambda=lam, random_state=0)`, fitted on `six` and `y6`.

</details>

<details>
<summary>💡 Hint 2</summary>

`left = six['vol_20d'].values <= 0.8` and `w_left = residuals[left].sum() / (left.sum() + lam)`. The model's forecasts are the mean plus the leaf value on each side: `np.allclose(one_tree.predict(six), y6.mean() + np.where(left, w_left, w_right), atol=0.00001)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
six = pd.DataFrame({'vol_20d': [0.4, 0.6, 0.8, 1.2, 1.6, 2.4]})
y6 = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])

left = six['vol_20d'].values <= 0.8
residuals = y6 - y6.mean()

for lam in [0, 1, 10]:
    one_tree = XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1, base_score=y6.mean(),
                            reg_lambda=lam, random_state=0)
    one_tree.fit(six, y6)
    w_left = residuals[left].sum() / (left.sum() + lam)
    w_right = residuals[~left].sum() / ((~left).sum() + lam)
    same = np.allclose(one_tree.predict(six), y6.mean() + np.where(left, w_left, w_right), atol=0.00001)
    print(lam, round(w_left, 4), round(w_right, 4), same)
```

±0.35 with λ = 0, the plain means of A2; ±0.2625 with λ = 1; ±0.081 with λ = 10. The model agrees each time. λ shrinks a leaf of n days to n/(n + λ) of its mean: a leaf of three days keeps three quarters at λ = 1, a leaf of 300 days almost all of it. The penalty reins in small leaves.

</details>

---

### E3 · Three libraries, one core  ★★☆☆☆  · revisits S11

Time 300 stumps at a learning rate of 0.1 on all of `columns` in `GradientBoostingRegressor`, `XGBRegressor` and `LGBMRegressor`, with `n_jobs=1` for the last two so that each uses one core. Store the seconds and the test RMSE of each in a dictionary of tuples.

In [ ]:
models = {
    'scikit-learn': ...,
    'XGBoost': ...,
    'LightGBM': ...,
}
results = {}
for name in models:
    ...

print(results)

<details>
<summary>💡 Hint 1</summary>

LightGBM makes a stump with `num_leaves=2`, and wants `verbose=-1`.

</details>

<details>
<summary>💡 Hint 2</summary>

In the loop: `start = time.perf_counter()`, fit `models[name]`, then `results[name] = (round(time.perf_counter() - start, 3), round(rmse(test['vol_next'], models[name].predict(test[columns])), 4))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
models = {
    'scikit-learn': GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=1, random_state=0),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=1, random_state=0, n_jobs=1),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, num_leaves=2, random_state=0,
                              n_jobs=1, verbose=-1),
}
results = {}
for name in models:
    start = time.perf_counter()
    models[name].fit(train[columns], train['vol_next'])
    seconds = time.perf_counter() - start
    results[name] = (round(seconds, 3), round(rmse(test['vol_next'], models[name].predict(test[columns])), 4))

print(results)
```

Here scikit-learn took 1.04 seconds, XGBoost 0.06 and LightGBM 0.03, with test RMSEs of 0.226, 0.221 and 0.233. The two libraries were about 16 and 34 times faster at the same job. The seconds depend on the machine; the ratios much less.

</details>

---

### E4 · LightGBM's leaves, stopped on 2022  ★★★☆☆  · revisits S2

Fit `LGBMRegressor` at a learning rate of 0.1 with up to 3,000 trees and `num_leaves` of 2, 4, 8 and 16, each stopped on `stop_rows` with the `early_stopping(50, verbose=False)` callback. Collect the number of trees, `best_iteration_`, and the best 2022 RMSE in a dictionary keyed by the number of leaves.

In [ ]:
by_leaves = {}
for leaves in [2, 4, 8, 16]:
    ...

print(by_leaves)

<details>
<summary>💡 Hint 1</summary>

`model.fit(fit_rows[columns], fit_rows['vol_next'], eval_set=[(stop_rows[columns], stop_rows['vol_next'])], callbacks=[early_stopping(50, verbose=False)])`.

</details>

<details>
<summary>💡 Hint 2</summary>

LightGBM records the squared error: `model.best_score_['valid_0']['l2'] ** 0.5` is the RMSE.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_leaves = {}
for leaves in [2, 4, 8, 16]:
    model = LGBMRegressor(n_estimators=3000, learning_rate=0.1, num_leaves=leaves, random_state=0, verbose=-1)
    model.fit(fit_rows[columns], fit_rows['vol_next'],
              eval_set=[(stop_rows[columns], stop_rows['vol_next'])],
              callbacks=[early_stopping(50, verbose=False)])
    by_leaves[leaves] = (model.best_iteration_, round(float(model.best_score_['valid_0']['l2'] ** 0.5), 4))

print(by_leaves)
```

48 stumps at 0.480; 15 trees of 4 leaves at 0.510; 13 of 8 at 0.541; 13 of 16 at 0.580. Bigger trees stop sooner and score worse on 2022, the pattern of C5 in another library. On this table the stump is the right size.

</details>

---

### E5 · Random rows and random columns  ★★★☆☆

`subsample` fits each tree on a random share of the rows, and `colsample_bytree` gives it a random share of the columns, as a forest does. Fit XGBoost stumps at a learning rate of 0.1, stopped on `stop_rows` 50 rounds after the best, with (subsample, colsample_bytree) of (1, 1), (0.5, 1), (1, 0.5) and (0.5, 0.5). Print the number of trees kept and the best 2022 RMSE for each pair.

In [ ]:
for rows_share, cols_share in [(1.0, 1.0), (0.5, 1.0), (1.0, 0.5), (0.5, 0.5)]:
    ...

<details>
<summary>💡 Hint 1</summary>

`XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1, subsample=rows_share, colsample_bytree=cols_share, early_stopping_rounds=50, random_state=0)`.

</details>

<details>
<summary>💡 Hint 2</summary>

Print `model.best_iteration + 1` and `round(model.best_score, 4)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for rows_share, cols_share in [(1.0, 1.0), (0.5, 1.0), (1.0, 0.5), (0.5, 0.5)]:
    model = XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1, subsample=rows_share,
                         colsample_bytree=cols_share, early_stopping_rounds=50, random_state=0)
    model.fit(fit_rows[columns], fit_rows['vol_next'],
              eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)
    print(rows_share, cols_share, model.best_iteration + 1, round(model.best_score, 4))
```

Without either, 244 trees and 0.464. Half the rows: 63 trees and 0.480; half the columns: 51 and 0.473; both: 75 and 0.482. Here the randomness made each tree noisier without making the sum better on 2022, so both settings stay at their defaults.

</details>

---

---

## F · Boosting a label

The credit table, as in the lecture: the depth, the log-odds behind a probability, the riskiest borrowers, two rankings drawn, and two ways to stop.

### F1 · Trees of depth 2  ★☆☆☆☆

Fit `XGBClassifier` with 100 trees of depth 2 at a learning rate of 0.1 on the credit table, and print its test AUC. The lecture's trees of depth 3 scored 0.777.

In [ ]:
xgb_c2 = ...
...

print(...)

<details>
<summary>💡 Hint</summary>

`roc_auc_score(c_test['default'], xgb_c2.predict_proba(c_test[c_columns])[:, 1])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_c2 = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=0)
xgb_c2.fit(c_train[c_columns], c_train['default'])

print(roc_auc_score(c_test['default'], xgb_c2.predict_proba(c_test[c_columns])[:, 1]))
```

0.775. One level less costs 0.002 of AUC: most of what trees of depth 3 find, trees of depth 2 find too.

</details>

---

### F2 · The log-odds behind a probability  ★★★☆☆  · revisits S8

A boosted classifier adds its trees on the log-odds scale. Fit XGBoost with 100 trees of depth 3 at a learning rate of 0.1, get the summed log-odds of the test borrowers with `predict(..., output_margin=True)`, turn them into probabilities with the sigmoid, and check them against `predict_proba`. Print the smallest and largest log-odds.

In [ ]:
xgb_c = ...
...
margin = ...
probability = ...

print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`margin = xgb_c.predict(c_test[c_columns], output_margin=True)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`probability = 1 / (1 + np.exp(-margin))`, then `np.allclose(probability, xgb_c.predict_proba(c_test[c_columns])[:, 1], atol=0.000001)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
xgb_c.fit(c_train[c_columns], c_train['default'])
margin = xgb_c.predict(c_test[c_columns], output_margin=True)
probability = 1 / (1 + np.exp(-margin))

print(np.allclose(probability, xgb_c.predict_proba(c_test[c_columns])[:, 1], atol=0.000001))
print(round(float(margin.min()), 2), round(float(margin.max()), 2))
```

`True`. The log-odds run from -4.05 to 2.00, which the sigmoid turns into probabilities from 0.017 to 0.88. The trees never see a probability: they move the log-odds, and the sigmoid is applied once, at the end.

</details>

---

### F3 · The 500 borrowers ranked riskiest  ★★★☆☆  · revisits S9

Rank the test borrowers by the probability of a default from XGBoost (100 trees of depth 3) and from a scaled logistic regression. For each model, take the 500 borrowers with the highest probability and print the share of them who defaulted, beside the share among all test borrowers.

In [ ]:
xgb_c = ...
...
logit = ...
...

for name, model in [('XGBoost', xgb_c), ('logistic regression', logit)]:
    ...

print('all test borrowers:', ...)

<details>
<summary>💡 Hint 1</summary>

`p = pd.Series(model.predict_proba(c_test[c_columns])[:, 1], index=c_test.index)` and `top = p.nlargest(500).index`.

</details>

<details>
<summary>💡 Hint 2</summary>

`c_test.loc[top, 'default'].mean()` is the share who defaulted among them.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
xgb_c.fit(c_train[c_columns], c_train['default'])
logit = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
logit.fit(c_train[c_columns], c_train['default'])

for name, model in [('XGBoost', xgb_c), ('logistic regression', logit)]:
    p = pd.Series(model.predict_proba(c_test[c_columns])[:, 1], index=c_test.index)
    top = p.nlargest(500).index
    print(name, round(c_test.loc[top, 'default'].mean(), 3))

print('all test borrowers:', round(c_test['default'].mean(), 3))
```

76.2 percent of XGBoost's 500 defaulted and 74.2 percent of logistic regression's, against 22.1 percent overall. At the very top the two models differ by 2 percentage points; the AUC gap, 0.777 against 0.748, is spread over the whole ranking.

</details>

---

### F4 · Two ROC curves on one axis  ★★☆☆☆  · revisits S8

Draw the ROC curves of a scaled logistic regression and of XGBoost (100 trees of depth 3) on the credit test rows, on one axis, with the diagonal of a ranking that knows nothing, a legend and a title.

In [ ]:
p_logit = ...
p_xgb = ...

fig, ax = plt.subplots(figsize=(5, 5))
...
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`fpr, tpr, _ = roc_curve(c_test['default'], p_logit)` and `ax.plot(fpr, tpr, label='logistic regression')`; the same for `p_xgb`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot([0, 1], [0, 1], linestyle='--', color='grey')` is the diagonal.

</details>

<details>
<summary>✅ Solution</summary>

```python
logit = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
logit.fit(c_train[c_columns], c_train['default'])
xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
xgb_c.fit(c_train[c_columns], c_train['default'])
p_logit = logit.predict_proba(c_test[c_columns])[:, 1]
p_xgb = xgb_c.predict_proba(c_test[c_columns])[:, 1]

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [('logistic regression', p_logit), ('XGBoost', p_xgb)]:
    fpr, tpr, _ = roc_curve(c_test['default'], p)
    ax.plot(fpr, tpr, label=name)
ax.plot([0, 1], [0, 1], linestyle='--', color='grey')
ax.set_xlabel('false positive rate')
ax.set_ylabel('true positive rate')
ax.legend()
ax.set_title('Credit test rows: two rankings', loc='left')
plt.show()
```

XGBoost's curve lies above logistic regression's over most of the range, and the difference in the area under them is small: 0.029. Both rise steeply at first, because the borrowers each model ranks riskiest mostly did default, as F3 found.

</details>

---

### F5 · Stop on the log-loss or on the AUC  ★★★☆☆  · revisits S9

Hold back a stratified fifth of the credit training rows, as the lecture did, and stop XGBoost (trees of depth 3, learning rate 0.1) on it twice: once on the log-loss, its default, and once with `eval_metric='auc'`. Print the number of trees each keeps and its test AUC.

In [ ]:
fit_c = ...
stop_c = ...

for metric in ['logloss', 'auc']:
    ...

<details>
<summary>💡 Hint 1</summary>

`fit_c, stop_c = train_test_split(c_train, test_size=0.2, random_state=0, stratify=c_train['default'])` fills both lines at once.

</details>

<details>
<summary>💡 Hint 2</summary>

`XGBClassifier(n_estimators=3000, learning_rate=0.1, max_depth=3, early_stopping_rounds=50, eval_metric=metric, random_state=0)`, fitted with `eval_set=[(stop_c[c_columns], stop_c['default'])]` and `verbose=False`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_c, stop_c = train_test_split(c_train, test_size=0.2, random_state=0, stratify=c_train['default'])

for metric in ['logloss', 'auc']:
    model = XGBClassifier(n_estimators=3000, learning_rate=0.1, max_depth=3,
                          early_stopping_rounds=50, eval_metric=metric, random_state=0)
    model.fit(fit_c[c_columns], fit_c['default'],
              eval_set=[(stop_c[c_columns], stop_c['default'])], verbose=False)
    auc = roc_auc_score(c_test['default'], model.predict_proba(c_test[c_columns])[:, 1])
    print(metric, model.best_iteration + 1, round(auc, 4))
```

107 trees on the log-loss and 91 on the AUC, with test AUCs of 0.7746 and 0.7747. The two yardsticks stop at different points and end with the same model, near enough: between 91 and 107 trees the AUC hardly moves.

</details>

---

---

## G · Reading a model

Importance on the credit table, measured four ways: from the cuts, by count and by gain; by shuffling a column; and by refitting without it. G4 and G5 run together.

### G1 · Gain or count  ★★☆☆☆

Fit XGBoost (100 trees of depth 3, learning rate 0.1) twice, once as it comes and once with `importance_type='weight'`, and print both `feature_importances_` as Series sorted from the largest. `'weight'` counts the cuts on each column; the default, `'gain'`, averages how much they lowered the loss.

In [ ]:
by_gain = ...
...
by_count = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0, importance_type='weight')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(model.feature_importances_, index=c_columns).sort_values(ascending=False)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_gain = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
by_gain.fit(c_train[c_columns], c_train['default'])
by_count = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0,
                         importance_type='weight')
by_count.fit(c_train[c_columns], c_train['default'])

print(pd.Series(by_gain.feature_importances_, index=c_columns).sort_values(ascending=False).round(3))
print(pd.Series(by_count.feature_importances_, index=c_columns).sort_values(ascending=False).round(3))
```

By gain, `late_now` comes first with 0.564 and `months_late` second; by count, `bill` comes first with 0.211 and `late_now` last with 0.084. The same 100 trees, two opposite rankings: `bill` has 16,700 different values in the training rows to cut at, and `late_now` has 9.

</details>

---

### G2 · Two columns of noise  ★★★☆☆  · revisits S3

Add two columns of noise to a copy of the credit training rows: `noise_num`, drawn from a normal distribution, and `noise_coin`, 0 or 1 at random, both from `np.random.default_rng(0)`, in that order. Fit `LGBMClassifier(random_state=0, verbose=-1)` on the seven columns and the two noise columns, and print its `feature_importances_`, the counts of cuts, sorted.

In [ ]:
noisy = ...
rng = ...
...
...
noisy_columns = ...

lgbm = ...
...

print(...)

<details>
<summary>💡 Hint 1</summary>

`rng = np.random.default_rng(0)`, then `rng.normal(size=len(noisy))` and `rng.integers(0, 2, size=len(noisy))`.

</details>

<details>
<summary>💡 Hint 2</summary>

Fit on `c_columns + ['noise_num', 'noise_coin']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
noisy = c_train.copy()
rng = np.random.default_rng(0)
noisy['noise_num'] = rng.normal(size=len(noisy))
noisy['noise_coin'] = rng.integers(0, 2, size=len(noisy))
noisy_columns = c_columns + ['noise_num', 'noise_coin']

lgbm = LGBMClassifier(random_state=0, verbose=-1)
lgbm.fit(noisy[noisy_columns], noisy['default'])

print(pd.Series(lgbm.feature_importances_, index=noisy_columns).sort_values(ascending=False))
```

`noise_num` is cut 540 times, more than any real column, and `noise_coin` 45 times, fewer than any. Both carry nothing. A column with thousands of values offers thousands of cuts, and some of them fit noise; a column with two values offers one. Counts of cuts measure opportunity as much as information.

</details>

---

### G3 · Permutation importance, by hand  ★★★★☆  · revisits S2

Fit XGBoost (100 trees of depth 3) and compute its test AUC. Then shuffle `months_late` in a copy of the test rows ten times, with `np.random.default_rng(k).permutation` for k from 0 to 9, and record the AUC each shuffle loses. Print the mean loss beside what `permutation_importance` reports for the same column with `n_repeats=10`.

In [ ]:
xgb_c = ...
...
base_auc = ...

losses = []
for k in range(10):
    ...

result = ...

print('by hand :', ...)
print('function:', ...)

<details>
<summary>💡 Hint 1</summary>

In the loop: `shuffled = c_test.copy()`, `shuffled['months_late'] = np.random.default_rng(k).permutation(shuffled['months_late'].values)`, then the AUC on `shuffled[c_columns]` and `losses.append(base_auc - ...)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`permutation_importance(xgb_c, c_test[c_columns], c_test['default'], scoring='roc_auc', n_repeats=10, random_state=0)`; `result.importances_mean[c_columns.index('months_late')]` is the column's entry.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
xgb_c.fit(c_train[c_columns], c_train['default'])
base_auc = roc_auc_score(c_test['default'], xgb_c.predict_proba(c_test[c_columns])[:, 1])

losses = []
for k in range(10):
    shuffled = c_test.copy()
    shuffled['months_late'] = np.random.default_rng(k).permutation(shuffled['months_late'].values)
    auc = roc_auc_score(c_test['default'], xgb_c.predict_proba(shuffled[c_columns])[:, 1])
    losses.append(base_auc - auc)

result = permutation_importance(xgb_c, c_test[c_columns], c_test['default'], scoring='roc_auc',
                                n_repeats=10, random_state=0)

print('by hand :', round(np.mean(losses), 4), ' sd', round(np.std(losses), 4))
print('function:', round(result.importances_mean[c_columns.index('months_late')], 4))
```

0.091 by hand and 0.087 from the function. The two use different shuffles, and the ten losses by hand vary with a standard deviation of 0.005, so a difference of that size is noise. The function runs the same loop for every column.

</details>

---

### G4 · Refit without each column  ★★★☆☆  · revisits S3

Drop-column importance refits the model without a column and measures what it loses. Fit XGBoost (100 trees of depth 3) on all seven columns as `full`. Then, for each column, refit on the other six and store the test AUC lost in a dictionary `drop_loss`. Print it as a Series sorted from the largest.

In [ ]:
full = ...
...
full_auc = ...

drop_loss = {}
for column in c_columns:
    ...

print(...)

<details>
<summary>💡 Hint 1</summary>

`rest = [c for c in c_columns if c != column]` holds the other six.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(drop_loss).sort_values(ascending=False)` sorts it.

</details>

<details>
<summary>✅ Solution</summary>

```python
full = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
full.fit(c_train[c_columns], c_train['default'])
full_auc = roc_auc_score(c_test['default'], full.predict_proba(c_test[c_columns])[:, 1])

drop_loss = {}
for column in c_columns:
    rest = [c for c in c_columns if c != column]
    model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
    model.fit(c_train[rest], c_train['default'])
    drop_loss[column] = full_auc - roc_auc_score(c_test['default'], model.predict_proba(c_test[rest])[:, 1])

print(pd.Series(drop_loss).sort_values(ascending=False).round(4))
```

`months_late` 0.015 and `late_now` 0.012 lead; the other five lose under 0.004. Every loss is far below the shuffle of G3 (0.087 for `months_late`), because a refit lets the other lateness column take over. Keep `full` and `drop_loss`: G5 draws them.

</details>

---

### G5 · Two importances, side by side  ★★☆☆☆  · revisits S3

Compute the permutation importance of `full` from G4 on the test rows, ten repeats, as a Series. Draw it and `drop_loss` from G4 as two horizontal bar charts side by side, each sorted, with a title on each.

In [ ]:
perm = ...

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`perm = pd.Series(permutation_importance(full, c_test[c_columns], c_test['default'], scoring='roc_auc', n_repeats=10, random_state=0).importances_mean, index=c_columns)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`s = perm.sort_values()` and `axes[0].barh(s.index, s.values)` draws one chart with the largest at the top; the same for `pd.Series(drop_loss)` on `axes[1]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
perm = pd.Series(permutation_importance(full, c_test[c_columns], c_test['default'], scoring='roc_auc',
                                        n_repeats=10, random_state=0).importances_mean, index=c_columns)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, s, title in [(axes[0], perm, 'AUC lost when shuffled'),
                     (axes[1], pd.Series(drop_loss), 'AUC lost by a refit without it')]:
    s = s.sort_values()
    ax.barh(s.index, s.values, color='#1c5cab')
    ax.set_title(title, loc='left')
fig.tight_layout()
plt.show()
```

Both rank `months_late` first and `late_now` second, and both put `age` last. The scales differ by a factor of about six: shuffling breaks a column for a model that relies on it, and a refit lets the model rely on something else.

</details>

---

### G6 · Groups of columns  ★★★☆☆  · revisits S3

Shuffle groups of columns together, with one permutation of the rows for the whole group, ten times each: the payment history (`late_now`, `months_late`), the amounts (`limit`, `bill`, `paid`, `utilisation`) and `age`. Print the mean AUC lost by XGBoost (100 trees of depth 3) for each group.

In [ ]:
xgb_c = ...
...
base_auc = ...
groups = {
    'payment history': ['late_now', 'months_late'],
    'amounts': ['limit', 'bill', 'paid', 'utilisation'],
    'age': ['age'],
}

group_loss = {}
for name in groups:
    ...

print(group_loss)

<details>
<summary>💡 Hint 1</summary>

`order = np.random.default_rng(k).permutation(len(c_test))`, then `shuffled[groups[name]] = c_test[groups[name]].values[order]` moves every column of the group with the same order.

</details>

<details>
<summary>💡 Hint 2</summary>

An inner loop over k from 0 to 9 collects the ten losses; store their mean in `group_loss[name]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
xgb_c.fit(c_train[c_columns], c_train['default'])
base_auc = roc_auc_score(c_test['default'], xgb_c.predict_proba(c_test[c_columns])[:, 1])
groups = {
    'payment history': ['late_now', 'months_late'],
    'amounts': ['limit', 'bill', 'paid', 'utilisation'],
    'age': ['age'],
}

group_loss = {}
for name in groups:
    losses = []
    for k in range(10):
        shuffled = c_test.copy()
        order = np.random.default_rng(k).permutation(len(c_test))
        shuffled[groups[name]] = c_test[groups[name]].values[order]
        auc = roc_auc_score(c_test['default'], xgb_c.predict_proba(shuffled[c_columns])[:, 1])
        losses.append(base_auc - auc)
    group_loss[name] = round(float(np.mean(losses)), 4)

print(group_loss)
```

The payment history together costs 0.210, the amounts 0.060 and `age` 0.002. Shuffled one at a time, the two lateness columns cost 0.087 and 0.038, less than the pair together: with one shuffled, the other still carries much of the same information. Shuffled together, nothing is left of it.

</details>

---

---

## H · Comparing models

Every classifier of the course on the credit table, timed; how many rows boosting needs; and two comparisons on the index table. H1 and H2, and H3 and H4, run together.

### H1 · Eight classifiers, one function  ★★★☆☆  · revisits S2

Write `evaluate(model)`: it fits the model on the credit training rows, times the fit, and returns the seconds and the test AUC. Run it on the eight models below and collect the results in a DataFrame `board`, one row per model, sorted by AUC. The forest and scikit-learn's `GradientBoosting` take a few seconds each.

In [ ]:
models = {
    'logistic regression': Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())]),
    'k-nearest neighbours': Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=25))]),
    'tree': DecisionTreeClassifier(max_depth=5, random_state=0),
    'random forest': RandomForestClassifier(n_estimators=100, min_samples_leaf=100, random_state=0),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, max_depth=3, random_state=0),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0, n_jobs=1),
    'LightGBM': LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=8, random_state=0,
                               n_jobs=1, verbose=-1),
}


def evaluate(model):
    """..."""
    ...


rows = []
for name in models:
    ...

board = ...
print(board)

<details>
<summary>💡 Hint 1</summary>

Inside `evaluate`: `start = time.perf_counter()`, fit, `seconds = time.perf_counter() - start`, then `return seconds, roc_auc_score(...)`.

</details>

<details>
<summary>💡 Hint 2</summary>

In the loop: `seconds, auc = evaluate(models[name])` and `rows.append({'model': name, 'seconds': round(seconds, 3), 'auc': round(auc, 4)})`. Then `pd.DataFrame(rows).set_index('model').sort_values('auc')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
models = {
    'logistic regression': Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())]),
    'k-nearest neighbours': Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=25))]),
    'tree': DecisionTreeClassifier(max_depth=5, random_state=0),
    'random forest': RandomForestClassifier(n_estimators=100, min_samples_leaf=100, random_state=0),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, max_depth=3, random_state=0),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0, n_jobs=1),
    'LightGBM': LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=8, random_state=0,
                               n_jobs=1, verbose=-1),
}


def evaluate(model):
    """Fit on the credit training rows; return the seconds and the test AUC."""
    start = time.perf_counter()
    model.fit(c_train[c_columns], c_train['default'])
    seconds = time.perf_counter() - start
    return seconds, roc_auc_score(c_test['default'], model.predict_proba(c_test[c_columns])[:, 1])


rows = []
for name in models:
    seconds, auc = evaluate(models[name])
    rows.append({'model': name, 'seconds': round(seconds, 3), 'auc': round(auc, 4)})

board = pd.DataFrame(rows).set_index('model').sort_values('auc')
print(board)
```

Logistic regression 0.748 and k-nearest neighbours 0.749, the scores of Session 9; a tree of depth 5 0.761; the forest 0.777; the four boosting implementations 0.776 to 0.777. Here the forest took 1.2 seconds and `GradientBoosting` 2.1, while the three binned implementations took 0.13 seconds or less for the same score. Keep `board`: H2 draws it.

</details>

---

### H2 · AUC against time  ★★☆☆☆  · revisits S3

Draw `board` from H1 as a scatter of AUC against seconds, with the x-axis on a log scale and each point labelled with its model's name.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.scatter(board['seconds'], board['auc'])` and `ax.set_xscale('log')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`for name in board.index: ax.annotate(name, (board.loc[name, 'seconds'], board.loc[name, 'auc']))` writes the labels.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(board['seconds'], board['auc'], color='#1c5cab')
for name in board.index:
    ax.annotate(name, (board.loc[name, 'seconds'], board.loc[name, 'auc']),
                textcoords='offset points', xytext=(5, 4), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('seconds to fit, one core')
ax.set_ylabel('test AUC')
ax.set_title('Eight classifiers on the credit table', loc='left')
plt.show()
```

The top left is the place to be, and the binned boosting libraries sit there: as accurate as the forest in a fraction of its time. Their labels overlap, which is the point. Logistic regression is the fastest of all and about 0.03 behind.

</details>

---

### H3 · How many rows boosting needs  ★★★☆☆  · revisits S5

Split a stratified fifth off the credit training rows as validation rows. Fit logistic regression (scaled) and XGBoost (100 trees of depth 3) on the first n of the remaining rows, for n in `[250, 500, 1000, 2000, 4000, 8000, 16800]`, and store both validation AUCs for each n in a DataFrame `curve` with columns `logistic` and `xgboost`.

In [ ]:
fit_c = ...
val_c = ...

rows = []
for n in [250, 500, 1000, 2000, 4000, 8000, 16800]:
    ...

curve = ...
print(curve)

<details>
<summary>💡 Hint 1</summary>

`fit_c, val_c = train_test_split(c_train, test_size=0.2, random_state=0, stratify=c_train['default'])`, then `fit_c.iloc[:n]` for the first n rows.

</details>

<details>
<summary>💡 Hint 2</summary>

Append `{'n': n, 'logistic': ..., 'xgboost': ...}` for each n, and `pd.DataFrame(rows).set_index('n')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_c, val_c = train_test_split(c_train, test_size=0.2, random_state=0, stratify=c_train['default'])

rows = []
for n in [250, 500, 1000, 2000, 4000, 8000, 16800]:
    part = fit_c.iloc[:n]
    logit = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
    logit.fit(part[c_columns], part['default'])
    xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
    xgb_c.fit(part[c_columns], part['default'])
    rows.append({'n': n,
                 'logistic': roc_auc_score(val_c['default'], logit.predict_proba(val_c[c_columns])[:, 1]),
                 'xgboost': roc_auc_score(val_c['default'], xgb_c.predict_proba(val_c[c_columns])[:, 1])})

curve = pd.DataFrame(rows).set_index('n')
print(curve.round(4))
```

Logistic regression is at 0.747 with 250 borrowers and never passes 0.761. XGBoost starts below it, at 0.736, passes it between 2,000 and 4,000 borrowers, and reaches 0.776 on all 16,800. A flexible model needs rows to pay for its flexibility; with a few hundred, logistic regression is the better choice. Keep `curve`: H4 draws it.

</details>

---

### H4 · Draw the two curves  ★★☆☆☆  · revisits S3

Draw both columns of `curve` from H3 against n on a log x-axis, with markers, axis labels and a legend.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(curve.index, curve['logistic'], marker='o', label='logistic regression')`, and the same for `curve['xgboost']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.set_xscale('log')` and `ax.legend()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(curve.index, curve['logistic'], marker='o', label='logistic regression')
ax.plot(curve.index, curve['xgboost'], marker='o', label='XGBoost')
ax.set_xscale('log')
ax.set_xlabel('training borrowers')
ax.set_ylabel('validation AUC')
ax.legend()
ax.set_title('More rows help the flexible model more', loc='left')
plt.show()
```

The lines cross once. To the left of the crossing the simpler model wins; to the right, boosting does, and its line is still rising at the end.

</details>

---

### H5 · Fold by fold  ★★★☆☆  · revisits S6

On the index table, compute the five fold RMSEs of ridge (a scaled pipeline with an alpha of 1000) and of 25 boosted stumps at a learning rate of 0.1, with `cross_val_score` on `folds`. Count with `zip` the folds where boosting has the lower RMSE, and print both lists.

In [ ]:
ridge_folds = ...
boost_folds = ...
boost_wins = ...

print(...)
print(...)
print(boost_wins)

<details>
<summary>💡 Hint 1</summary>

`-cross_val_score(Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))]), train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error')` gives the five RMSEs.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for r, b in zip(ridge_folds, boost_folds) if b < r)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ridge_folds = -cross_val_score(Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))]),
                               train[columns], train['vol_next'], cv=folds,
                               scoring='neg_root_mean_squared_error')
boost_folds = -cross_val_score(GradientBoostingRegressor(n_estimators=25, learning_rate=0.1, max_depth=1,
                                                         random_state=0),
                               train[columns], train['vol_next'], cv=folds,
                               scoring='neg_root_mean_squared_error')
boost_wins = sum(1 for r, b in zip(ridge_folds, boost_folds) if b < r)

print(ridge_folds.round(3))
print(boost_folds.round(3))
print(boost_wins)
```

Ridge wins all five: 0.476, 0.472, 1.127, 0.327, 0.397 against 0.634, 0.505, 1.254, 0.460, 0.495, and `boost_wins` is 0. Boosting does not lose on average because of one bad fold; it loses on every one, including fold 3, which holds 2020.

</details>

---

### H6 · A forecast used as a ranking  ★★★★☆  · revisits S8

On the index table, call a day busy when `vol_next` is above the median of the training days. Fit 219 boosted stumps at a learning rate of 0.1, the number 2022 chose in C4, on `vol_next` itself, and `XGBClassifier` with 100 stumps on the busy label. Compute the test AUC of the regression's forecast as a score for the label, of the classifier's probability, and of `vol_20d` itself.

In [ ]:
median = ...
busy_train = ...
busy_test = ...

regression = ...
...
classifier = ...
...

print('forecast  :', ...)
print('classifier:', ...)
print('vol_20d   :', ...)

<details>
<summary>💡 Hint 1</summary>

`busy_train = (train['vol_next'] > median).astype(int)`, and the same on the test days with the training median.

</details>

<details>
<summary>💡 Hint 2</summary>

`roc_auc_score` takes any score that ranks the days: `roc_auc_score(busy_test, regression.predict(test[columns]))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
median = train['vol_next'].median()
busy_train = (train['vol_next'] > median).astype(int)
busy_test = (test['vol_next'] > median).astype(int)

regression = GradientBoostingRegressor(n_estimators=219, learning_rate=0.1, max_depth=1, random_state=0)
regression.fit(train[columns], train['vol_next'])
classifier = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
classifier.fit(train[columns], busy_train)

print('forecast  :', roc_auc_score(busy_test, regression.predict(test[columns])))
print('classifier:', roc_auc_score(busy_test, classifier.predict_proba(test[columns])[:, 1]))
print('vol_20d   :', roc_auc_score(busy_test, test['vol_20d']))
```

The forecast of the number scores 0.733, the classifier 0.646 and `vol_20d` alone 0.614. The classifier only ever saw whether a day was above the median; the regression saw by how much, which is more information. Here, modelling the number and ranking by its forecast beat modelling the label made from it.

</details>

---

---

## I · When it goes wrong

Five mistakes that come with boosting. Four stop with an error, and one runs quietly with a forecast that is slightly off. In each, the first cell shows the mistake and the second is for your fix.

### I1 · Early stopping with nothing to watch  ★☆☆☆☆

The cell below asks XGBoost to stop early and raises an error. Read its last line. Then fit the model in the second cell so that it stops on 2022, and print `best_iteration`.

In [ ]:
xgb_bad = XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1, early_stopping_rounds=50,
                       random_state=0)
xgb_bad.fit(fit_rows[columns], fit_rows['vol_next'])

In [ ]:
xgb_good = ...
...

print(...)

<details>
<summary>💡 Hint 1</summary>

Early stopping needs rows to score each round on.

</details>

<details>
<summary>💡 Hint 2</summary>

`eval_set=[(stop_rows[columns], stop_rows['vol_next'])]` in `fit`, with `verbose=False` to keep the output short.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_good = XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1, early_stopping_rounds=50,
                        random_state=0)
xgb_good.fit(fit_rows[columns], fit_rows['vol_next'],
             eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)

print(xgb_good.best_iteration)
```

`ValueError: Must have at least 1 validation dataset for early stopping`, then 243. Without an `eval_set` there is nothing to stop on.

</details>

---

### I2 · One tree short  ★★☆☆☆

The cell below rebuilds the stopped model with a fixed number of trees, and its `assert` fails. Find the mistake and fix it in the second cell, with the `assert` passing.

In [ ]:
xgb_stop = XGBRegressor(n_estimators=3000, learning_rate=0.1, max_depth=1, early_stopping_rounds=50,
                        random_state=0)
xgb_stop.fit(fit_rows[columns], fit_rows['vol_next'],
             eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)
refit = XGBRegressor(n_estimators=xgb_stop.best_iteration, learning_rate=0.1, max_depth=1, random_state=0)
refit.fit(fit_rows[columns], fit_rows['vol_next'])
assert np.allclose(refit.predict(test[columns]), xgb_stop.predict(test[columns]))

In [ ]:
refit = ...
...
...

print(...)

<details>
<summary>💡 Hint</summary>

`best_iteration` counts rounds from 0. How many trees does round 243 mean?

</details>

<details>
<summary>✅ Solution</summary>

```python
refit = XGBRegressor(n_estimators=xgb_stop.best_iteration + 1, learning_rate=0.1, max_depth=1,
                     random_state=0)
refit.fit(fit_rows[columns], fit_rows['vol_next'])
assert np.allclose(refit.predict(test[columns]), xgb_stop.predict(test[columns]))

print(rmse(test['vol_next'], refit.predict(test[columns])))
```

`AssertionError`. Round 243 is the 244th tree, so the refit needs `best_iteration + 1` trees. One tree short, the forecasts differ by up to 0.0006 and the test RMSE is 0.22437 instead of 0.22459. Nothing warned; only the `assert` noticed.

</details>

---

### I3 · No importances to read  ★★☆☆☆

`HistGradientBoostingRegressor` is scikit-learn's binned boosting. The cell below asks it for importances and raises an error. In the second cell, measure its importance on the test days with `permutation_importance` instead, and print the three columns it loses most without.

In [ ]:
hist = HistGradientBoostingRegressor(max_iter=200, max_depth=2, random_state=0)
hist.fit(train[columns], train['vol_next'])
print(hist.feature_importances_)

In [ ]:
hist = ...
...
result = ...
importance = ...

print(...)

<details>
<summary>💡 Hint 1</summary>

Refit `hist` as in the cell above. Then `permutation_importance(hist, test[columns], test['vol_next'], scoring='neg_root_mean_squared_error', n_repeats=10, random_state=0)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(result.importances_mean, index=columns).nlargest(3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
hist = HistGradientBoostingRegressor(max_iter=200, max_depth=2, random_state=0)
hist.fit(train[columns], train['vol_next'])
result = permutation_importance(hist, test[columns], test['vol_next'],
                                scoring='neg_root_mean_squared_error', n_repeats=10, random_state=0)
importance = pd.Series(result.importances_mean, index=columns)

print(importance.nlargest(3).round(4))
```

`AttributeError`: the binned models of scikit-learn have no `feature_importances_`. Shuffled on the test days, the model loses most without `XOM_vol` (0.0290 of RMSE), then `ret_60d` and `MSFT_vol`. Permutation importance works for any model that predicts, which is one more reason to prefer it.

</details>

---

### I4 · A label written as words  ★★☆☆☆  · revisits S8

The cell below fits XGBoost on a label of words, `'jump'` when `vol_next` is more than 1.5 times `vol_20d` and `'quiet'` otherwise, and raises an error. Read what it expected. Then fit the same model in the second cell on the same label as 0 and 1, with 1 for a jump, and print its test AUC.

In [ ]:
words = np.where(train['vol_next'] > 1.5 * train['vol_20d'], 'jump', 'quiet')
xgb_words = XGBClassifier(n_estimators=100, max_depth=1, random_state=0)
xgb_words.fit(train[columns], words)

In [ ]:
jump_train = ...
jump_test = ...
xgb_jump = ...
...

print(...)

<details>
<summary>💡 Hint</summary>

`(train['vol_next'] > 1.5 * train['vol_20d']).astype(int)` is 1 for a jump; the same on `test`.

</details>

<details>
<summary>✅ Solution</summary>

```python
jump_train = (train['vol_next'] > 1.5 * train['vol_20d']).astype(int)
jump_test = (test['vol_next'] > 1.5 * test['vol_20d']).astype(int)
xgb_jump = XGBClassifier(n_estimators=100, max_depth=1, random_state=0)
xgb_jump.fit(train[columns], jump_train)

print(roc_auc_score(jump_test, xgb_jump.predict_proba(test[columns])[:, 1]))
```

`ValueError: Invalid classes inferred from unique values of y. Expected: [0 1], got ['jump' 'quiet']`. scikit-learn's own classifiers accept words; XGBoost wants the classes numbered from 0. With numbers, the test AUC is 0.821, on 36 jumps among 482 test days.

</details>

---

### I5 · A generator is not a list  ★★☆☆☆  · revisits S2

The cell below wants the test forecast after 50 stumps and raises an error. Fix it in the second cell, and print the test RMSE after 50 stumps.

In [ ]:
boost = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
boost.fit(train[columns], train['vol_next'])
after_50 = boost.staged_predict(test[columns])[49]

In [ ]:
boost = ...
...
after_50 = ...

print(...)

<details>
<summary>💡 Hint</summary>

`staged_predict` hands out one forecast at a time, in order. `list(...)` collects them all, and then `[49]` is the fiftieth.

</details>

<details>
<summary>✅ Solution</summary>

```python
boost = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
boost.fit(train[columns], train['vol_next'])
after_50 = list(boost.staged_predict(test[columns]))[49]

print(rmse(test['vol_next'], after_50))
```

`TypeError: 'generator' object is not subscriptable`. A generator produces each stage only when asked, which saves memory on long runs; `list()` stores them all. After 50 stumps the test RMSE is 0.226.

</details>

---

---

## J · A target with nothing in it

Whether the index rises tomorrow, from its last five daily returns. Daily returns are close to unpredictable, which makes this a clean test of what a flexible model does when there is nothing to find. J2 to J6 use the table from J1.

### J1 · Five past returns and tomorrow's direction  ★★★☆☆  · revisits S4

From `rets['SPY']`, the index's daily returns in percent, build a DataFrame `direction` with `lag_0` (today's return) to `lag_4` (the return four days earlier), made with `shift` in a loop, and `up_next`: 1 when tomorrow's return is positive. Drop the first five rows and the last one, which are incomplete, and split at the end of 2022 into `d_train` and `d_test`. Print the share of up days in each.

In [ ]:
spy = rets['SPY']
direction = pd.DataFrame()
for k in range(5):
    ...
...
...

d_train = ...
d_test = ...
lags = ['lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_4']

print(...)

<details>
<summary>💡 Hint 1</summary>

`direction['lag_' + str(k)] = spy.shift(k)` in the loop. Tomorrow is `spy.shift(-1)`, so `direction['up_next'] = (spy.shift(-1) > 0).astype(int)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`direction = direction.iloc[5:-1]` drops the incomplete rows; then `.loc[:'2022-12-31']` and `.loc['2023-01-01':]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
spy = rets['SPY']
direction = pd.DataFrame()
for k in range(5):
    direction['lag_' + str(k)] = spy.shift(k)
direction['up_next'] = (spy.shift(-1) > 0).astype(int)
direction = direction.iloc[5:-1]

d_train = direction.loc[:'2022-12-31']
d_test = direction.loc['2023-01-01':]
lags = ['lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_4']

print(len(d_train), len(d_test), round(d_train['up_next'].mean(), 3), round(d_test['up_next'].mean(), 3))
```

2,008 training days and 501 test days; 53.9 percent of the training days and 57.5 percent of the test days were followed by a rise. Every column is known at the close, and the label is the next day's move: nothing in the table looks ahead.

</details>

---

### J2 · Logistic regression first  ★★☆☆☆  · revisits S8

Fit a scaled logistic regression of `up_next` on the five lags, and print its AUC on the training days and on the test days.

In [ ]:
logit = ...
...

print('train:', ...)
print('test :', ...)

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])`, fitted on `d_train[lags]` and `d_train['up_next']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
logit = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
logit.fit(d_train[lags], d_train['up_next'])

print('train:', roc_auc_score(d_train['up_next'], logit.predict_proba(d_train[lags])[:, 1]))
print('test :', roc_auc_score(d_test['up_next'], logit.predict_proba(d_test[lags])[:, 1]))
```

0.532 on the training days and 0.471 on the test days, below the 0.5 of a coin. With five columns a straight line cannot fit much noise, and there was little else to fit.

</details>

---

### J3 · Three hundred deeper trees  ★★☆☆☆

Fit `XGBClassifier` with 300 trees of depth 4 at a learning rate of 0.1 on the same columns, and print both AUCs.

In [ ]:
xgb_dir = ...
...

print('train:', ...)
print('test :', ...)

<details>
<summary>💡 Hint</summary>

`XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, random_state=0)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_dir = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, random_state=0)
xgb_dir.fit(d_train[lags], d_train['up_next'])

print('train:', roc_auc_score(d_train['up_next'], xgb_dir.predict_proba(d_train[lags])[:, 1]))
print('test :', roc_auc_score(d_test['up_next'], xgb_dir.predict_proba(d_test[lags])[:, 1]))
```

0.981 on the training days and 0.487 on the test days. The trees found patterns in 2,008 days that rank them almost perfectly, and none of it carried over.

</details>

---

### J4 · What early stopping says  ★★★☆☆

Fit the same model on the days up to the end of 2021 and stop it on 2022, with `eval_metric='auc'` and `early_stopping_rounds=50`, from up to 1,000 trees. Print `best_iteration`, the best 2022 AUC and the test AUC.

In [ ]:
d_fit = ...
d_stop = ...

xgb_dir = ...
...

print(...)

<details>
<summary>💡 Hint 1</summary>

`d_train.loc[:'2021-12-31']` and `d_train.loc['2022-01-01':]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=4, early_stopping_rounds=50, eval_metric='auc', random_state=0)`, with `eval_set=[(d_stop[lags], d_stop['up_next'])]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
d_fit = d_train.loc[:'2021-12-31']
d_stop = d_train.loc['2022-01-01':]

xgb_dir = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=4,
                        early_stopping_rounds=50, eval_metric='auc', random_state=0)
xgb_dir.fit(d_fit[lags], d_fit['up_next'], eval_set=[(d_stop[lags], d_stop['up_next'])], verbose=False)

print(xgb_dir.best_iteration, round(xgb_dir.best_score, 4),
      round(roc_auc_score(d_test['up_next'], xgb_dir.predict_proba(d_test[lags])[:, 1]), 4))
```

`best_iteration` is 3, so the model keeps 4 trees, with a 2022 AUC of 0.575 and a test AUC of 0.535. Early stopping cannot turn noise into signal, but it stops the fitting of it: 4 trees instead of 300. Random scores on these 501 test days give AUCs with a standard deviation of 0.026, so 0.535 is less than two standard deviations from a coin's 0.5.

</details>

---

### J5 · A label shuffled on purpose  ★★★★☆  · revisits S2

Shuffle `up_next` in a copy of `d_train` with `np.random.default_rng(0).permutation`, so that no column can carry any information about it. Fit J3's model on the shuffled label, and print its AUC on the shuffled training days and on the real test days.

In [ ]:
shuffled = ...
...
xgb_noise = ...
...

print('train:', ...)
print('test :', ...)

<details>
<summary>💡 Hint 1</summary>

`shuffled = d_train.copy()`, then `shuffled['up_next'] = np.random.default_rng(0).permutation(shuffled['up_next'].values)`.

</details>

<details>
<summary>💡 Hint 2</summary>

Score the training AUC against the shuffled label, and the test AUC against `d_test['up_next']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
shuffled = d_train.copy()
shuffled['up_next'] = np.random.default_rng(0).permutation(shuffled['up_next'].values)
xgb_noise = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, random_state=0)
xgb_noise.fit(shuffled[lags], shuffled['up_next'])

print('train:', roc_auc_score(shuffled['up_next'], xgb_noise.predict_proba(shuffled[lags])[:, 1]))
print('test :', roc_auc_score(d_test['up_next'], xgb_noise.predict_proba(d_test[lags])[:, 1]))
```

0.976 on the training days, against 0.981 with the real label, and 0.502 on the test days. A model this flexible fits a label it cannot possibly predict almost as well as the real one. A high training score shows capacity, not signal.

</details>

---

### J6 · Both AUCs, tree by tree  ★★☆☆☆  · revisits S3

Fit 300 trees of depth 4 on `d_fit` from J4, with `eval_metric='auc'` and two `eval_set` entries, the fitting days first and 2022 second, and no early stopping. Draw both AUC paths from `evals_result()` against the number of trees, with a dashed line at 0.5 and a legend.

In [ ]:
xgb_paths = ...
...
paths = ...

fig, ax = plt.subplots(figsize=(8, 3))
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`evals_result()['validation_0']['auc']` belongs to the first entry of `eval_set`, the fitting days, and `['validation_1']['auc']` to the second.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.axhline(0.5, linestyle='--', color='grey')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
xgb_paths = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, eval_metric='auc', random_state=0)
xgb_paths.fit(d_fit[lags], d_fit['up_next'],
              eval_set=[(d_fit[lags], d_fit['up_next']), (d_stop[lags], d_stop['up_next'])], verbose=False)
paths = xgb_paths.evals_result()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, 301), paths['validation_0']['auc'], label='fitting days, 2015 to 2021')
ax.plot(range(1, 301), paths['validation_1']['auc'], label='2022')
ax.axhline(0.5, linestyle='--', color='grey')
ax.set_xlabel('trees')
ax.set_ylabel('AUC')
ax.legend()
ax.set_title('Tomorrow up or down: what the trees learn', loc='left')
plt.show()
```

The fitting days climb from 0.606 to 0.986 while 2022 drifts down from 0.564 to 0.538. When the two lines move apart from the first trees on, the trees are fitting noise.

</details>

---

---

## K · Five small cases

Each of these stands on its own and needs nothing from earlier in the notebook beyond the setup cell. They keep functions, loops, dictionaries and NumPy in working order, in this session's setting.

### K1 · A boosted model as a sum  ★★☆☆☆  · revisits S2

The first three stumps that boosting fits to the six days at a learning rate of 0.5 are below, each as (cut, value at or below the cut, value above it). The model starts from 0.95. Write `predict(x)`, which adds 0.5 times each stump's value to 0.95 for one day, and print the forecast for each of the six days.

In [ ]:
stumps = [(1.0, -0.35, 0.35), (1.0, -0.175, 0.175), (1.4, -0.0688, 0.1375)]
days = [0.4, 0.6, 0.8, 1.2, 1.6, 2.4]


def predict(x):
    """..."""
    ...


for x in days:
    print(x, ...)

<details>
<summary>💡 Hint 1</summary>

Start `total = 0.95`, loop `for cut, left, right in stumps:`, and add `0.5 * left` if `x <= cut`, otherwise `0.5 * right`.

</details>

<details>
<summary>💡 Hint 2</summary>

Return `total`, and print `round(predict(x), 4)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
stumps = [(1.0, -0.35, 0.35), (1.0, -0.175, 0.175), (1.4, -0.0688, 0.1375)]
days = [0.4, 0.6, 0.8, 1.2, 1.6, 2.4]


def predict(x):
    """The forecast of three boosted stumps for one day."""
    total = 0.95
    for cut, left, right in stumps:
        if x <= cut:
            total = total + 0.5 * left
        else:
            total = total + 0.5 * right
    return total


for x in days:
    print(x, round(predict(x), 4))
```

0.6531 for the three calm days, 1.1781 for 1.2 and 1.2812 for 1.6 and 2.4: the forecast after A4's third step. The third stump cut at 1.4, so it was the first to split the busy side.

</details>

---

### K2 · Bins, by hand  ★★★☆☆  · revisits S2

XGBoost and LightGBM cut a column only at bin edges set by its quantiles. Make four bins of `vol_20d` on the training days of the index table by hand: compute the 25th, 50th and 75th percentiles with `np.quantile`, then write `bin_of(value, edges)` that returns 0, 1, 2 or 3 with `if` and `elif`. Count the days in each bin, and print the mean `vol_next` of each bin.

In [ ]:
values = train['vol_20d']
edges = ...


def bin_of(value, edges):
    """..."""
    ...


bins = []
for v in values:
    ...

counts = ...
means = ...

print(edges)
print(counts)
print(means)

<details>
<summary>💡 Hint 1</summary>

`edges = np.quantile(values, [0.25, 0.5, 0.75])`. In `bin_of`: `if value <= edges[0]: return 0`, then `elif value <= edges[1]: return 1`, and so on.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(bins).value_counts().sort_index()` counts them, and `train['vol_next'].groupby(np.array(bins)).mean()` gives the means.

</details>

<details>
<summary>✅ Solution</summary>

```python
values = train['vol_20d']
edges = np.quantile(values, [0.25, 0.5, 0.75])


def bin_of(value, edges):
    """The bin, 0 to 3, that a value falls in."""
    if value <= edges[0]:
        return 0
    elif value <= edges[1]:
        return 1
    elif value <= edges[2]:
        return 2
    return 3


bins = []
for v in values:
    bins.append(bin_of(v, edges))

counts = pd.Series(bins).value_counts().sort_index()
means = train['vol_next'].groupby(np.array(bins)).mean()

print(edges.round(3))
print(counts)
print(means.round(3))
```

Edges at 0.524, 0.794 and 1.271 percent, 473 or 474 days in each bin, and a mean `vol_next` of 0.641, 0.753, 1.068, 1.467: next month rises with this month at every step. XGBoost does the same with 256 bins per column, which is why it only ever tries 255 cuts per column, however many days there are.

</details>

---

### K3 · XGBoost's gain of a cut  ★★★★☆  · revisits S2

XGBoost scores a cut with $G_L^2/(H_L+\lambda) + G_R^2/(H_R+\lambda) - (G_L+G_R)^2/(H_L+H_R+\lambda)$, where $G$ is the sum of the gradients $f - y$ in a box and $H$, for squared error, its number of days. (The library also halves it, which changes no ranking.) On the six days, with the forecast at their mean and $\lambda = 1$, compute the gain of each cut after 0.4, 0.6, 0.8, 1.2 and 1.6 with a loop, store it in a dictionary, and print the best cut. Then do the same with $\lambda = 0$ and compare the best gain with the fall in RSS that cut gives, from 0.815 to 0.080.

In [ ]:
x = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
y = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
g = ...

for lam in [1, 0]:
    gains = {}
    for cut in [0.4, 0.6, 0.8, 1.2, 1.6]:
        ...
    print(lam, gains)
    print('best:', ...)

<details>
<summary>💡 Hint 1</summary>

`g = y.mean() - y`, the gradient of ½(y − f)² at the mean. For a cut, `left = x <= cut`, `GL = g[left].sum()`, `HL = left.sum()`, and the same with `~left`.

</details>

<details>
<summary>💡 Hint 2</summary>

`gains[cut] = round(GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR)**2 / (HL + HR + lam), 4)`, and `max(gains, key=gains.get)` is the best cut.

</details>

<details>
<summary>✅ Solution</summary>

```python
x = np.array([0.4, 0.6, 0.8, 1.2, 1.6, 2.4])
y = np.array([0.5, 0.6, 0.7, 1.2, 1.5, 1.2])
g = y.mean() - y

for lam in [1, 0]:
    gains = {}
    for cut in [0.4, 0.6, 0.8, 1.2, 1.6]:
        left = x <= cut
        GL, GR = g[left].sum(), g[~left].sum()
        HL, HR = left.sum(), (~left).sum()
        gains[cut] = round(float(GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR)**2 / (HL + HR + lam)), 4)
    print(lam, gains)
    print('best:', max(gains, key=gains.get))
```

With λ = 1 the cut after 0.8 wins with 0.551, ahead of 0.6 and 1.2 at 0.341 each. With λ = 0 its gain is 0.735, exactly the fall in RSS from 0.815 to 0.080: without the penalty, XGBoost's rule is the tree rule of Session 11. The penalty lowers every gain and small boxes most: the cut after 0.4, which leaves one day alone, falls from 0.243 to 0.135.

</details>

---

### K4 · How much a leaf keeps  ★★☆☆☆  · revisits S2

With the penalty $\lambda$, a leaf of $n$ days keeps $n/(n + \lambda)$ of its plain mean residual (E2). Build a DataFrame with one row per $n$ in `[1, 5, 25, 100, 500]` and one column per $\lambda$ in `[1, 10, 100]`, filled with that share to three decimals.

In [ ]:
sizes = [1, 5, 25, 100, 500]
shares = {}
for lam in [1, 10, 100]:
    ...

kept = ...
print(kept)

<details>
<summary>💡 Hint 1</summary>

For each λ, build a list of `round(n / (n + lam), 3)` over the sizes, and store it as `shares[lam]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.DataFrame(shares, index=sizes)` makes the table.

</details>

<details>
<summary>✅ Solution</summary>

```python
sizes = [1, 5, 25, 100, 500]
shares = {}
for lam in [1, 10, 100]:
    shares[lam] = [round(n / (n + lam), 3) for n in sizes]

kept = pd.DataFrame(shares, index=sizes)
print(kept)
```

At λ = 1 every leaf of 25 days or more keeps over 96 percent; at λ = 100 a leaf of 5 days keeps 4.8 percent and a leaf of 500 keeps 83.3 percent. On a table of 1,894 days with stumps, λ = 1 changes almost nothing; the small gap between XGBoost's stumps and scikit-learn's came from the bins.

</details>

---

### K5 · Settings as dictionaries  ★★★☆☆  · revisits S2

Three settings for XGBoost are stored below as dictionaries. `XGBRegressor(random_state=0, **settings)` passes a dictionary's entries as arguments. Fit each on `fit_rows`, score it on `stop_rows`, and print the settings with the lowest 2022 RMSE.

In [ ]:
candidates = [
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 1},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2},
    {'n_estimators': 50, 'learning_rate': 0.3, 'max_depth': 1},
]
rmse_2022 = []
for settings in candidates:
    ...

best = ...
print(rmse_2022)
print(best)

<details>
<summary>💡 Hint 1</summary>

In the loop: `model = XGBRegressor(random_state=0, **settings)`, fit on `fit_rows[columns]`, and append the RMSE on `stop_rows`.

</details>

<details>
<summary>💡 Hint 2</summary>

`candidates[int(np.argmin(rmse_2022))]` is the best dictionary.

</details>

<details>
<summary>✅ Solution</summary>

```python
candidates = [
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 1},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2},
    {'n_estimators': 50, 'learning_rate': 0.3, 'max_depth': 1},
]
rmse_2022 = []
for settings in candidates:
    model = XGBRegressor(random_state=0, **settings)
    model.fit(fit_rows[columns], fit_rows['vol_next'])
    rmse_2022.append(round(rmse(stop_rows['vol_next'], model.predict(stop_rows[columns])), 4))

best = candidates[int(np.argmin(rmse_2022))]
print(rmse_2022)
print(best)
```

0.466, 0.557 and 0.474: 200 stumps at a rate of 0.1 win. A grid search does this for every combination in a dictionary of lists, and `**` is how a chosen dictionary of settings becomes a model.

</details>

---

## 🏁 Done

You boosted the six days by hand and as a function, showed that five stumps can memorise them, checked that the trees are fitted to the negative gradient for a number and for a label, watched the training error fall at every stump, chose the number of trees on 2022 and on the folds, rebuilt XGBoost's early stopping and its leaf values, timed three libraries, boosted the credit label, measured importance four ways, compared every classifier of the course, and gave a flexible model a target with nothing in it.

The case is the last part of the risk report: boosting for its volatility forecast, held to everything the report has fitted since Part 4, and then every model of the course side by side, ending with the report as it would go out.